<a href="https://colab.research.google.com/github/DABMASTER-Brought-me-into-this/NeuroLearn/blob/main/SlideMDCNNImageClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import numpy.lib.stride_tricks as lst
import matplotlib.pyplot as plt
import math
import random
import re
import cv2
import csv
from google.colab.patches import cv2_imshow
%matplotlib inline

In [2]:
# Hyperparameters
MAX_WORD_LEN = 32
BATCH_SIZE = 6
WIN_SIZE_L1 = (5, 5)
NUM_FEATURES_L1 = 128
MAX_POOLING_SHRINK_L1 = 4
WIN_SIZE_L2 = (5, 5)
NUM_FEATURES_L2 = 64
MAX_POOLING_SHRINK_L2 = 4
WIN_SIZE_L3 = (5, 5)
NUM_FEATURES_L3 = 8
MAX_POOLING_SHRINK_L3 = 2

In [3]:
# # Opening the File
# dataset = []
# with open('/content/slidemdimageds.csv', mode='r', newline='') as file:
#     reader = csv.reader(file)
#     for row in reader:
#         if len(row) == 3:
#           dataset.append(row)

In [4]:
# # Seperating Inputs and Outputs
# last_num_int = lambda a: int(a[-1])
# inputs_seperation = lambda a: a[:2]
# Y = list(map(last_num_int, dataset))
# inputs = list(map(inputs_seperation, dataset))

In [5]:
# # Preprocessing Text
# stoi = {chr(i): (i - ord('a') + 1) for i in range(ord('a'), ord('z') + 1)}
# stoi['.'] = 0
# itos = {ix: letter for letter, ix in stoi.items()}

In [6]:
# # Converting All Words Into Numbers
# def tok(word):
#   word = re.sub('[^a-z]', '', word.lower())
#   letters = list(word)
#   enc_letters = list(map(lambda a: stoi[a], letters))
#   if len(enc_letters) < MAX_WORD_LEN:
#     enc_letters.extend([0] * (MAX_WORD_LEN - len(enc_letters)))
#   return enc_letters

In [7]:
# # Replacing All Text with Tokenized Words
# inputs = [[input[0], tok(input[1])] for input in inputs]

In [8]:
# Preprocessing Image
images = []
for i in range(0,6):
    img_path = f"image_{i}.jpeg"
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        resized_img = cv2.resize(img, (256, 256))
        images.append(resized_img)

X = np.array(images).astype(np.float32)
X /= np.max(X)
Y = np.array([1,0,1,1,1,0])

In [9]:
# PARAPARAMETER!!!
W1 = np.random.randn(*WIN_SIZE_L1, NUM_FEATURES_L1)
g1 = np.ones(NUM_FEATURES_L1)
b1 = np.zeros_like(g1)
W2 = np.random.randn(*WIN_SIZE_L2, NUM_FEATURES_L1, NUM_FEATURES_L2)
g2 = np.ones(NUM_FEATURES_L2)
b2 = np.zeros_like(g2)
W3 = np.random.randn(*WIN_SIZE_L2, NUM_FEATURES_L2, NUM_FEATURES_L3)
g3 = np.ones(NUM_FEATURES_L3)
b3 = np.zeros_like(g3)

In [10]:
### Forward Pass

## Layer 1

# Padding + Conv
pad1 = tuple((size-1)//2 for size in WIN_SIZE_L1) # Same Padding Calcs
img_padded1 = np.pad(X, ((0,0), pad1, pad1), "constant", constant_values=0)
win_l1 = lst.sliding_window_view(img_padded1, window_shape = WIN_SIZE_L1, axis = (1,2))
win_l1r = (np.ascontiguousarray(win_l1)).reshape(-1, win_l1.shape[-2] * win_l1.shape[-1])
preln1 = win_l1r @ W1.reshape(-1, W1.shape[-1])
preln1 = preln1.reshape(BATCH_SIZE, X.shape[1], X.shape[1], NUM_FEATURES_L1)
# Layer Norm
lnmean1 = np.mean(preln1, axis = 1, keepdims = True)
lnvar1 = np.var(preln1, axis = 1, keepdims = True)
lnraw1 = (preln1 - lnmean1)/np.sqrt(lnvar1 + 1e-5)
ln1 = g1 * lnraw1 + b1
# ReLu
rl1 = np.maximum(np.zeros_like(ln1), ln1)
# Max Pooling
rl1 = rl1.reshape(BATCH_SIZE, NUM_FEATURES_L1, rl1.shape[1], -1)
wins_1 = lst.sliding_window_view(rl1, window_shape=(MAX_POOLING_SHRINK_L1,MAX_POOLING_SHRINK_L1), axis = (2,3))
wins_1 = np.ascontiguousarray(wins_1)
img_pieced1 = (wins_1[:, :, 0::MAX_POOLING_SHRINK_L1, 0::MAX_POOLING_SHRINK_L1, :, :]).reshape(wins_1.shape[:-4] + (rl1.shape[-1]//MAX_POOLING_SHRINK_L1,-1,MAX_POOLING_SHRINK_L1**2))
pooled_img1 = np.max(img_pieced1, axis=-1)

## Layer 2
# Padding + Conv
pad2 = tuple((size-1)//2 for size in WIN_SIZE_L2)
img_padded2 = np.pad(pooled_img1, ((0,0), (0,0), pad2, pad2), "constant", constant_values=0)
win_l2 = lst.sliding_window_view(img_padded2, window_shape = WIN_SIZE_L2, axis = (2,3))
win_l2r = np.swapaxes(np.expand_dims(np.ascontiguousarray(win_l2), -1), 1, -1)
win_l2r = win_l2r.squeeze(1)
win_l2r = win_l2r.reshape(-1, win_l2r.shape[-2] * win_l2r.shape[-1] * win_l2r.shape[-3])
preln2 = win_l2r @ W2.reshape(-1, W2.shape[-1])
preln2 = preln2.reshape(BATCH_SIZE, pooled_img1.shape[2], pooled_img1.shape[2], NUM_FEATURES_L2)
# Layer Norm
lnmean2 = np.mean(preln2, axis = -1, keepdims = True)
lnvar2 = np.var(preln2, axis = -1, keepdims = True)
lnraw2 = (preln2 - lnmean2)/np.sqrt(lnvar2 + 1e-5)
ln2 = g2 * lnraw2 + b2
# ReLu
rl2 = np.maximum(np.zeros_like(ln2), ln2)
# Max Pooling
rl2 = rl2.reshape(BATCH_SIZE, NUM_FEATURES_L2, rl2.shape[1], -1)
wins_2 = lst.sliding_window_view(rl2, window_shape=(MAX_POOLING_SHRINK_L2,MAX_POOLING_SHRINK_L2), axis = (2,3))
wins_2 = np.ascontiguousarray(wins_2)
img_pieced2 = (wins_2[:, :, 0::MAX_POOLING_SHRINK_L2, 0::MAX_POOLING_SHRINK_L2, :, :]).reshape(wins_2.shape[:-4] + (rl2.shape[-1]//MAX_POOLING_SHRINK_L2,-1,MAX_POOLING_SHRINK_L2**2))
pooled_img2 = np.max(img_pieced2, axis=-1)

## Layer 3
# Padding + Conv
pad3 = tuple((size-1)//2 for size in WIN_SIZE_L3)
img_padded3 = np.pad(pooled_img2, ((0,0), (0,0), pad3, pad3), "constant", constant_values=0)
win_l3 = lst.sliding_window_view(img_padded3, window_shape = WIN_SIZE_L3, axis = (2,3))
win_l3r = np.swapaxes(np.expand_dims(np.ascontiguousarray(win_l3), -1), 1, -1)
win_l3r = win_l3r.squeeze(axis=1)
win_l3r = win_l3r.reshape(-1, win_l3r.shape[-2] * win_l3r.shape[-1] * win_l3r.shape[-3])
preln3 = win_l3r @ W3.reshape(-1, W3.shape[-1])
preln3 = preln3.reshape(BATCH_SIZE, pooled_img2.shape[2], pooled_img2.shape[2], NUM_FEATURES_L3)
# Layer Norm
lnmean3 = np.mean(preln3, axis = -1, keepdims = True)
lnvar3 = np.var(preln3, axis = -1, keepdims = True)
lnraw3 = (preln3 - lnmean3)/np.sqrt(lnvar3 + 1e-5)
ln3 = g3 * lnraw3 + b3
# ReLu
rl3 = np.maximum(np.zeros_like(ln3), ln3)
# Max Pooling
rl3 = rl3.reshape(BATCH_SIZE, NUM_FEATURES_L3, rl3.shape[1], -1)
wins_3 = lst.sliding_window_view(rl3, window_shape=(MAX_POOLING_SHRINK_L3,MAX_POOLING_SHRINK_L3), axis = (2,3))
wins_3 = np.ascontiguousarray(wins_3)
img_pieced3 = (wins_3[:, :, 0::MAX_POOLING_SHRINK_L3, 0::MAX_POOLING_SHRINK_L3, :, :]).reshape(wins_3.shape[:-4] + (rl3.shape[-1]//MAX_POOLING_SHRINK_L3,-1,MAX_POOLING_SHRINK_L3**2))
pooled_img3 = np.max(img_pieced3, axis=-1)
pooled_img3.shape

(6, 8, 8, 8)

In [11]:
# # Me Self Inventing Max Pooling
# size = random.randint(1, 100)
# size = size if size % 2 == 0 else size + 1
# img = np.random.randn(size**2).reshape(size,size)
# wins = lst.sliding_window_view(img, window_shape=(2,2), axis = (0,1))
# img_pieced = (wins[0::2, 0::2, :, :]).reshape(-1, 2 * 2)
# max_pooling = np.max(img_pieced, axis = 1)
# max_pooling = max_pooling.reshape(-1, int(np.sqrt(max_pooling.shape[0])))